# Bulk PBMC clock and comparison with PeriClock

Paper: Fig. 6i–j and Extended Data Fig. 10. Inputs: original donor-level bulk training DataFrame; nonzero stable-feature table; final PeriClock artifact; annotated HNSCC and NSCLC cohorts; matched NSCLC raw counts. Environment: Python/Scanpy, NumPy, pandas, SciPy, scikit-learn, joblib, Matplotlib, seaborn and adjustText. Outputs: fitted bulk pipeline and feature metadata, cohort predictions/statistics and gene-weight comparison tables/plots under `OUTPUT_DIR`. All runtime variables are initialized in this notebook. Large-data, training and clinical execution remain unverified.

Original source indices below are **zero-based**. Original cell hashes are retained in cell metadata. Outputs were cleared; inclusion does not imply successful execution.

## Required resources and configuration

`PeriClock-bulk_features_matrix.pkl`, bundled nonzero coefficients, final PeriClock model, prepared clinical HNSCC/NSCLC data, exact matched NSCLC raw-count AnnData and licensed Arial fonts; [training matrices](../RESOURCE_DOWNLOAD.md#4-training-feature-matrices) and [immunotherapy inputs](../RESOURCE_DOWNLOAD.md#7-immunotherapy-datasets).

Download locations, sizes, SHA256, versions, permissions and publication status are maintained once in [RESOURCE_DOWNLOAD.md](../RESOURCE_DOWNLOAD.md). Individual GEO/GSM sources and sample choices are in the paper’s Supplementary Table 1. Clock models are uploaded to [Hugging Face Models](https://huggingface.co/LeyiZhang1/PeriClock); [Dataset uploads](https://huggingface.co/datasets/LeyiZhang1/PeriClock-data) are in progress. Both repositories are currently private. Consult the resource guide for per-file availability; donor-level public redistribution remains subject to review.

Set `PERICLOCK_RESOURCE_DIR` for external data and `PERICLOCK_MODEL_DIR` for models, or override individual file paths in the next cell. Keep outputs separate from inputs. Start a fresh kernel and run in order.


In [ ]:
from pathlib import Path
import os
# Start Jupyter from within this GitHub checkout.
PROJECT_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                     if (p / "periclock").is_dir() and (p / "pyproject.toml").is_file()), None)
if PROJECT_ROOT is None:
    raise RuntimeError("Start the notebook kernel from within the PeriClock code checkout")
RESOURCE_DIR = Path(os.environ.get("PERICLOCK_RESOURCE_DIR", "~/periclock-resources")).expanduser().resolve()
MODEL_DIR = Path(os.environ.get("PERICLOCK_MODEL_DIR", str(RESOURCE_DIR / "models"))).expanduser().resolve()
# Each individual file-path environment variable still overrides these defaults.
MODEL_PATH = Path(os.environ.get("PERICLOCK_PERTURBATION_MODEL", str(MODEL_DIR / "PeriClock.pkl"))).expanduser().resolve()
HNSCC_H5AD = Path(os.environ.get("PERICLOCK_HNSCC_H5AD", str(RESOURCE_DIR / "datasets/HNSCC_Blood_celltypist_annotated.h5ad"))).expanduser().resolve()
NSCLC_H5AD = Path(os.environ.get("PERICLOCK_NSCLC_H5AD", str(RESOURCE_DIR / "datasets/GSE306542_Symphony_annotated.h5ad"))).expanduser().resolve()
BULK_TRAINING_TABLE = Path(os.environ.get("PERICLOCK_BULK_TRAINING_TABLE", str(RESOURCE_DIR / "datasets/PeriClock-bulk_features_matrix.pkl"))).expanduser().resolve()
NONZERO_FEATURES = Path(os.environ.get("PERICLOCK_NONZERO_FEATURES", str(PROJECT_ROOT / "resources/features/stable_feature_coef_no0.csv"))).expanduser().resolve()
NSCLC_COUNTS = Path(os.environ.get("PERICLOCK_NSCLC_COUNTS", str(RESOURCE_DIR / "datasets/GSE306542_processed.h5ad"))).expanduser().resolve()
FONT_DIR = Path(os.environ.get("PERICLOCK_FONT_DIR", "/path/to/Arial")).expanduser().resolve()
OUTPUT_DIR = Path(os.environ.get("PERICLOCK_TUTORIAL_OUTPUT", str(PROJECT_ROOT / "outputs/10_bulk_clock"))).expanduser().resolve()
# Input files remain read-only; all relative output paths below use this directory.
for input_path in [MODEL_PATH, HNSCC_H5AD, NSCLC_H5AD, BULK_TRAINING_TABLE, NONZERO_FEATURES, NSCLC_COUNTS]:
    if not input_path.is_file():
        raise FileNotFoundError(f"Set the input path in this cell: {input_path}")
if not FONT_DIR.is_dir():
    raise FileNotFoundError("Set PERICLOCK_FONT_DIR to a directory containing licensed Arial fonts")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
os.chdir(OUTPUT_DIR)

## Bulk clock restricted to PeriClock genes

Source: cells 26, 28–29 and 31–33. The saved `PeriClock-bulk_features_matrix.pkl` must be the original donor-level mean **log1p(CP10K)** expression DataFrame with an `Age` column. This is not a raw-count pseudobulk or a newly normalized substitute. The nonzero stable-feature table defines the original 131 candidate genes. The 80/20 split, StandardScaler, ElasticNetCV grid and hyperparameters remain unchanged. Model files are written only into this tutorial's output directory; downloaded models are not overwritten. This training section has not been executed.

In [ ]:
for folder in ["data", "model", "result", "fig"]:
    Path(folder).mkdir(parents=True, exist_ok=True)

In [ ]:
import pandas as pd
import joblib
import scanpy as sc
import numpy as np

In [ ]:
periclock_fea = pd.read_csv(str(NONZERO_FEATURES))
periclock_fea

In [ ]:
import pandas as pd
import numpy as np

# 仅保留表达特征
expr_fea = periclock_fea[
    periclock_fea["FeatureType"].eq("Expression")
].copy()

# 提取唯一基因
peri_genes = expr_fea["Gene"].dropna().astype(str).drop_duplicates().tolist()

print("PeriClock expression features:", len(expr_fea))
print("Unique genes:", len(peri_genes))
print("Genes:", peri_genes)

In [ ]:
df_bulk = pd.read_pickle(str(BULK_TRAINING_TABLE))
# 检查基因覆盖情况
available_genes = [g for g in peri_genes if g in df_bulk.columns]
missing_genes = [g for g in peri_genes if g not in df_bulk.columns]

print("PeriClock genes:", len(peri_genes))
print("Available genes:", len(available_genes))
print("Missing genes:", missing_genes)

if missing_genes:
    raise ValueError("部分PeriClock基因不在Bulk特征矩阵中，请先检查")

# 构建新模型的特征矩阵
df_bulk_perigenes = df_bulk[["Age"] + available_genes].copy()

print("Shape:", df_bulk_perigenes.shape)
display(df_bulk_perigenes.head())

# 保存
df_bulk_perigenes.to_pickle("data/df_bulk_perigenes.pkl")

In [ ]:
import joblib
import json
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import ElasticNetCV
from sklearn.pipeline import make_pipeline
from sklearn.metrics import r2_score, mean_absolute_error
from scipy.stats import pearsonr

# ============================================================
# 1. 数据准备
# ============================================================
X = df_bulk_perigenes.drop(columns="Age")
y = df_bulk_perigenes["Age"].astype(float)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# ============================================================
# 2. 模型训练
# ============================================================
model_perigenes = make_pipeline(
    StandardScaler(),
    ElasticNetCV(
        l1_ratio=[0.1, 0.3, 0.5, 0.7, 0.9, 0.95, 1.0],
        cv=5,
        random_state=42,
        max_iter=20000,
        n_jobs=4
    )
)

model_perigenes.fit(X_train, y_train)

# ============================================================
# 3. 性能评价
# ============================================================
y_pred = model_perigenes.predict(X_test)

metrics = {
    "R2": r2_score(y_test, y_pred),
    "MAE": mean_absolute_error(y_test, y_pred),
    "PCC": pearsonr(y_test, y_pred).statistic,
    "MAD": np.median(np.abs(y_test.to_numpy() - y_pred))
}

print("\n===== BulkClock-PeriGenes =====")
for k, v in metrics.items():
    print(f"{k}: {v:.4f}")

en = model_perigenes.named_steps["elasticnetcv"]

print("\nBest alpha:", en.alpha_)
print("Best l1_ratio:", en.l1_ratio_)
print("Selected genes:", np.sum(en.coef_ != 0))
print("Iterations:", en.n_iter_)
print("Dual gap:", en.dual_gap_)

In [ ]:
# 保存完整Pipeline
joblib.dump(model_perigenes, "model/bulk_clock_perigenes_model.pkl", compress=3)

# 保存必要参数
en = model_perigenes.named_steps["elasticnetcv"]

model_info = {
    "model_name": "BulkClock-PeriGenes",
    "n_samples": len(df_bulk_perigenes),
    "n_features": X.shape[1],
    "feature_names": X.columns.tolist(),
    "best_alpha": float(en.alpha_),
    "best_l1_ratio": float(en.l1_ratio_),
    "n_selected_features": int(np.sum(en.coef_ != 0)),
    "selected_features": X.columns[en.coef_ != 0].tolist(),
    "metrics": {k: float(v) for k, v in metrics.items()},
    "random_state": 42,
    "test_size": 0.2,
    "cv": 5
}

with open("model/bulk_clock_perigenes_model_info.json", "w") as f:
    json.dump(model_info, f, indent=4, ensure_ascii=False)

print("Model saved.")

## NSCLC and HNSCC applications (Fig. 6i–j)

Source: cells 39–42, 44–47 and 49. Models are reread from the files generated above. NSCLC requires the original raw-count AnnData **and** the matched QC/annotated cell list: the original normalization is performed before selecting the model genes. HNSCC uses the already normalized annotated AnnData. The original response definitions, aggregation, tests and plots are retained. No clinical dictionary is included. Restricted inputs and generated clinical outputs must remain outside the public repository.

In [ ]:
import os
import json
import joblib
import numpy as np
import pandas as pd
import scanpy as sc
from scipy import sparse

# ============================================================
# 1. 加载 BulkClock-PeriGenes 模型
# ============================================================
model_dir = str(OUTPUT_DIR / "model")

model = joblib.load(f"{model_dir}/bulk_clock_perigenes_model.pkl")

with open(f"{model_dir}/bulk_clock_perigenes_model_info.json") as f:
    model_info = json.load(f)

feature_names = model_info["feature_names"]

print("Model: BulkClock-PeriGenes")
print("Required genes:", len(feature_names))

# ============================================================
# 2. 检查原始数据的基因覆盖和细胞匹配
# ============================================================
adata_raw = sc.read_h5ad(str(NSCLC_COUNTS))
adata_nsclc = sc.read_h5ad(str(NSCLC_H5AD))
assert adata_raw.obs_names.is_unique
assert adata_raw.var_names.is_unique
assert adata_nsclc.obs_names.is_unique
assert adata_nsclc.obs_names.isin(adata_raw.obs_names).all()

available_genes = [g for g in feature_names if g in adata_raw.var_names]
missing_genes = [g for g in feature_names if g not in adata_raw.var_names]

print("\n===== Gene coverage =====")
print(f"Required genes: {len(feature_names)}")
print(f"Available genes: {len(available_genes)}")
print(f"Missing genes: {len(missing_genes)}")
print("Missing:", missing_genes)

# ============================================================
# 3. 计算QC后细胞的标准化表达
# ============================================================
# 使用QC后的细胞，保留原始数据的全部基因
adata_perigenes = adata_raw[adata_nsclc.obs_names, :].copy()

# 原始X应为未标准化counts
sc.pp.normalize_total(adata_perigenes, target_sum=1e4)
sc.pp.log1p(adata_perigenes)

# 标准化后提取模型相关基因
adata_perigenes = adata_perigenes[:, available_genes].copy()

# 使用QC后的临床信息
adata_perigenes.obs = adata_nsclc.obs.reindex(adata_perigenes.obs_names).copy()

print("\nProcessed AnnData:")
print(adata_perigenes)

# ============================================================
# 4. 按患者构建 Bulk 表达矩阵
# ============================================================
obs = adata_perigenes.obs
patient_col = "patient_ids"

if obs[patient_col].isna().any():
    raise ValueError("存在缺失患者ID")

# 确保每名患者对应唯一年龄和响应类型
for col in ["age", "response_type_9weeks"]:
    if (obs.groupby(patient_col, observed=True)[col].nunique(dropna=False) > 1).any():
        raise ValueError(f"同一患者存在多个{col}")

codes, patients = pd.factorize(obs[patient_col], sort=True)
n_cells = adata_perigenes.n_obs
n_patients = len(patients)
cell_counts = np.bincount(codes, minlength=n_patients)

group = sparse.csr_matrix(
    (np.ones(n_cells, dtype=np.float32), (codes, np.arange(n_cells))),
    shape=(n_patients, n_cells)
)

bulk = group @ adata_perigenes.X

if sparse.issparse(bulk):
    bulk = bulk.toarray()

bulk = np.asarray(bulk, dtype=np.float32) / cell_counts[:, None]

df_bulk_perigenes_nsclc = pd.DataFrame(
    bulk,
    index=pd.Index(patients, name="patient_ids"),
    columns=available_genes
)

# ============================================================
# 5. 补充临床信息
# ============================================================
clinical = obs.groupby(patient_col, observed=True).first().reindex(patients)

df_bulk_perigenes_nsclc.insert(
    0, "Age", pd.to_numeric(clinical["age"]).to_numpy()
)

df_bulk_perigenes_nsclc.insert(
    1, "Response", clinical["response_type_9weeks"].astype(str).to_numpy()
)

print("\n===== BulkClock-PeriGenes NSCLC features =====")
print("Shape:", df_bulk_perigenes_nsclc.shape)
display(df_bulk_perigenes_nsclc.head())

# ============================================================
# 6. 保存特征矩阵
# ============================================================
output_file = str(OUTPUT_DIR / "data/bulk_perigenes_nsclc_features.pkl")
os.makedirs(os.path.dirname(output_file), exist_ok=True)

df_bulk_perigenes_nsclc.to_pickle(output_file)
print("Saved:", output_file)

In [ ]:
# ============================================================
# 7. 检查模型特征完整性
# ============================================================
missing = [g for g in feature_names if g not in df_bulk_perigenes_nsclc.columns]

if missing:
    raise ValueError(f"缺少 {len(missing)} 个模型基因: {missing}")

# ============================================================
# 8. 预测年龄
# ============================================================
X_nsclc = df_bulk_perigenes_nsclc[feature_names]
pred_age = model.predict(X_nsclc)

results_nsclc_perigenes = pd.DataFrame({
    "Age": df_bulk_perigenes_nsclc["Age"],
    "Predicted_Age": pred_age,
    "Delta_Age": pred_age - df_bulk_perigenes_nsclc["Age"].to_numpy(),
    "Response": df_bulk_perigenes_nsclc["Response"]
}, index=df_bulk_perigenes_nsclc.index)

results_nsclc_perigenes["Response"] = results_nsclc_perigenes["Response"].map({
    "PR": "Responder",
    "PD": "Non-responder"
})

if results_nsclc_perigenes["Response"].isna().any():
    raise ValueError("存在未识别的治疗响应类型")

display(results_nsclc_perigenes)

results_nsclc_perigenes.to_csv(
    str(OUTPUT_DIR / "result/bulk_perigenes_nsclc_predictions.csv")
)

In [ ]:
from scipy.stats import mannwhitneyu

stats_rows = []

for var in ["Age", "Predicted_Age", "Delta_Age"]:
    pr = results_nsclc_perigenes.loc[
        results_nsclc_perigenes["Response"] == "Responder", var
    ].dropna()

    pd_group = results_nsclc_perigenes.loc[
        results_nsclc_perigenes["Response"] == "Non-responder", var
    ].dropna()

    stat, p = mannwhitneyu(pr, pd_group, alternative="two-sided", method="exact")

    stats_rows.append({
        "Variable": var,
        "PR_N": len(pr),
        "PD_N": len(pd_group),
        "PR_Mean": pr.mean(),
        "PD_Mean": pd_group.mean(),
        "PR_Median": pr.median(),
        "PD_Median": pd_group.median(),
        "P_value": p
    })

stats_nsclc_perigenes = pd.DataFrame(stats_rows)
display(stats_nsclc_perigenes)

stats_nsclc_perigenes.to_csv(
    str(OUTPUT_DIR / "result/bulk_perigenes_nsclc_statistics.csv"), index=False
)

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
import matplotlib.font_manager as fm
import seaborn as sns
from scipy.stats import mannwhitneyu

# ============================================================
# 1. Arial 字体及全局参数
# ============================================================
font_dir = str(FONT_DIR)
if os.path.isdir(font_dir):
    for f in os.listdir(font_dir):
        if f.lower().endswith((".ttf", ".otf")):
            fm.fontManager.addfont(os.path.join(font_dir, f))

mpl.rcParams.update({
    "font.family": "Arial",
    "font.sans-serif": ["Arial"],
    "font.size": 12,
    "font.weight": "bold",
    "axes.labelsize": 14,
    "axes.labelweight": "bold",
    "axes.titlesize": 14,
    "axes.titleweight": "bold",
    "xtick.labelsize": 12,
    "ytick.labelsize": 12,
    "axes.linewidth": 1.2,
    "pdf.fonttype": 42,
    "ps.fonttype": 42
})

# ============================================================
# 2. 数据准备
# ============================================================
df = results_nsclc_perigenes.copy()

# 兼容 PR/PD 或 Responder/Non-responder
df["Response"] = df["Response"].replace({
    "PR": "Responder",
    "PD": "Non-responder"
})

order = ["Responder", "Non-responder"]
palette = {
    "Responder": "#4E9F91",
    "Non-responder": "#D97575"
}

plot_vars = [
    ("Age", "Chronological age (years)"),
    ("Predicted_Age", "Predicted age (years)"),
    ("Delta_Age", "Delta age (years)")
]

# ============================================================
# 3. 绘制三联图
# ============================================================
fig, axes = plt.subplots(1, 3, figsize=(12, 4.6))

stats_rows = []

for ax, (var, ylabel) in zip(axes, plot_vars):

    pr = df.loc[df["Response"] == "Responder", var].dropna()
    pd_group = df.loc[df["Response"] == "Non-responder", var].dropna()

    # 双侧 Mann–Whitney U 检验
    stat, p = mannwhitneyu(
        pr, pd_group, alternative="two-sided", method="exact"
    )

    stats_rows.append({
        "Variable": var,
        "PR_N": len(pr),
        "PD_N": len(pd_group),
        "PR_Mean": pr.mean(),
        "PD_Mean": pd_group.mean(),
        "PR_Median": pr.median(),
        "PD_Median": pd_group.median(),
        "P_value": p
    })

    # 箱线图
    sns.boxplot(
        data=df, x="Response", y=var,
        hue="Response", order=order, palette=palette,
        dodge=False, width=0.48, showfliers=False,
        linewidth=1.3, legend=False, ax=ax,
        boxprops={"alpha": 0.55}
    )

    # 每位患者的散点
    sns.stripplot(
        data=df, x="Response", y=var,
        hue="Response", order=order, palette=palette,
        dodge=False, jitter=0.10, size=8,
        edgecolor="black", linewidth=0.5,
        alpha=0.95, legend=False, ax=ax
    )

    # Delta Age 零参考线
    if var == "Delta_Age":
        ax.axhline(0, color="#888888", ls="--", lw=1.2, zorder=0)

    ax.set_title(f"P = {p:.3g}", pad=12)
    ax.set_xlabel("")
    ax.set_ylabel(ylabel)
    ax.set_xticklabels(["Responder", "Non-responder"], rotation=15)

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.tick_params(direction="out", length=4, width=1.1)

plt.tight_layout(w_pad=2.5)

# ============================================================
# 4. 保存图片和统计结果
# ============================================================
output_dir = str(OUTPUT_DIR)
os.makedirs(output_dir, exist_ok=True)

plt.savefig(f"{output_dir}/bulk_perigenes_nsclc_response.pdf", bbox_inches="tight")
plt.savefig(f"{output_dir}/bulk_perigenes_nsclc_response.png", dpi=300, bbox_inches="tight")
plt.show()

stats_df = pd.DataFrame(stats_rows)
display(stats_df)

stats_df.to_csv(
    f"{output_dir}/bulk_perigenes_nsclc_response_statistics.csv",
    index=False
)

In [ ]:
adata = sc.read_h5ad(str(HNSCC_H5AD))
adata

In [ ]:
import os
import json
import joblib
import numpy as np
import pandas as pd
from scipy import sparse
from scipy.stats import pearsonr
from sklearn.metrics import r2_score, mean_absolute_error

# ============================================================
# 1. 加载模型，提取2834个非零系数基因
# ============================================================
model_dir = "model"
model = joblib.load(os.path.join(model_dir, "bulk_clock_perigenes_model.pkl"))

with open(os.path.join(model_dir, "bulk_clock_perigenes_model_info.json"), "r") as f:
    model_info = json.load(f)

scaler = model.named_steps["standardscaler"]
en = model.named_steps["elasticnetcv"]

all_genes = np.array(model_info["feature_names"])
mask = en.coef_ != 0

selected_genes = all_genes[mask].tolist()
selected_coef = en.coef_[mask].copy()
selected_mean = scaler.mean_[mask].copy()
selected_scale = scaler.scale_[mask].copy()
intercept = float(en.intercept_)

print(f"Original genes: {len(all_genes)}")
print(f"Selected genes: {len(selected_genes)}")
print(f"Best alpha: {en.alpha_:.6f}")
print(f"Best l1_ratio: {en.l1_ratio_}")

# ============================================================
# 2. 构建个体水平 Bulk 表达矩阵
# ============================================================
def build_bulk_features(adata, donor_col, age_col, feature_names):
    obs = adata.obs[[donor_col, age_col]].copy()

    if obs.isna().any().any():
        raise ValueError(f"{donor_col} 或 {age_col} 存在缺失值")

    obs[age_col] = pd.to_numeric(obs[age_col], errors="raise")

    if not adata.var_names.is_unique:
        raise ValueError("adata.var_names 存在重复基因，请先处理")

    missing = [g for g in feature_names if g not in adata.var_names]

    print(f"Required genes: {len(feature_names)}")
    print(f"Available genes: {len(feature_names) - len(missing)}")

    if missing:
        print(f"Missing genes ({len(missing)}):", missing[:30])
        raise ValueError("缺少模型选中的基因，暂不自动填0")

    # 检查一个个体是否对应多个年龄
    age_count = obs.groupby(donor_col, observed=True)[age_col].nunique()

    if (age_count > 1).any():
        raise ValueError(f"同一 {donor_col} 对应多个年龄")

    # 只提取选中的基因
    gene_idx = adata.var_names.get_indexer(feature_names)
    X = adata.X[:, gene_idx]

    if sparse.issparse(X):
        X = X.tocsr().astype(np.float32)
    else:
        X = np.asarray(X, dtype=np.float32)

    # 构建 donor 分组矩阵
    donor_codes, donors = pd.factorize(obs[donor_col], sort=True)
    n_cells, n_donors = len(donor_codes), len(donors)
    counts = np.bincount(donor_codes, minlength=n_donors)

    group = sparse.csr_matrix(
        (
            np.ones(n_cells, dtype=np.float32),
            (donor_codes, np.arange(n_cells))
        ),
        shape=(n_donors, n_cells)
    )

    # 按个体计算所有细胞的平均表达
    bulk = group @ X

    if sparse.issparse(bulk):
        bulk = bulk.toarray()

    bulk = np.asarray(bulk, dtype=np.float32) / counts[:, None]

    df_bulk = pd.DataFrame(
        bulk,
        index=pd.Index(donors, name=donor_col),
        columns=feature_names
    )

    age_map = obs.groupby(donor_col, observed=True)[age_col].first()
    df_bulk.insert(0, "Age", age_map.reindex(df_bulk.index).to_numpy())

    print(f"Cells: {n_cells:,}")
    print(f"Individuals: {n_donors}")
    print(f"Genes: {len(feature_names)}")
    print(f"Bulk shape: {df_bulk.shape}")

    return df_bulk

# ============================================================
# 3. 只使用2834个基因预测年龄
# ============================================================
def evaluate_bulk_clock(df_bulk, dataset_name):
    X = df_bulk[selected_genes].to_numpy(dtype=np.float64)
    y_true = df_bulk["Age"].to_numpy(dtype=float)

    # 应用训练时保存的 StandardScaler 参数
    X_scaled = (X - selected_mean) / selected_scale

    # Elastic Net 原始系数和截距
    y_pred = X_scaled @ selected_coef + intercept

    results = pd.DataFrame({
        "Age": y_true,
        "Predicted_Age": y_pred,
        "Delta_Age": y_pred - y_true
    }, index=df_bulk.index)

    metrics = {
        "Dataset": dataset_name,
        "N": len(results),
        "R2": r2_score(y_true, y_pred),
        "MAE": mean_absolute_error(y_true, y_pred),
        "PCC": pearsonr(y_true, y_pred).statistic,
        "MAD": np.median(np.abs(y_true - y_pred))
    }

    print(f"\n===== {dataset_name} =====")
    print(f"N   = {metrics['N']}")
    print(f"R²  = {metrics['R2']:.4f}")
    print(f"MAE = {metrics['MAE']:.4f}")
    print(f"PCC = {metrics['PCC']:.4f}")
    print(f"MAD = {metrics['MAD']:.4f}")

    return results, metrics

# ============================================================
# 4. 预测
# ============================================================
print("\nProcessing HNSCC...")

df_bulk_hnscc = build_bulk_features(
    adata,
    donor_col="sample",
    age_col="age",
    feature_names=selected_genes
)

results_hnscc, metrics_hnscc = evaluate_bulk_clock(
    df_bulk_hnscc, "HNSCC"
)

# ============================================================
# 6. 整理并保存
# ============================================================
metrics_df = pd.DataFrame([metrics_hnscc])

# 保存预测结果，保留个体ID
results_hnscc.to_csv("bulk_clock_hnscc_predictions.csv", index=True)
metrics_df.to_csv("bulk_clock_external_metrics.csv", index=False)

# 保存聚合后的bulk表达矩阵
df_bulk_hnscc.to_pickle("bulk_clock_hnscc_features.pkl")

print("\n===== External validation summary =====")
display(metrics_df)

print("\nHNSCC predictions:")
display(results_hnscc.head())


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import permutation_test

pred = results_hnscc.copy()

# ============================================================
# 1. 提取样本临床信息
# ============================================================
clinical_cols = [
    'sample', 'patient', 'timepoint', 'age',
    'MPR', 'pathologic_response', 'radiographic_response',
    'treatment'
]

meta = adata.obs[clinical_cols].copy()
meta['sample'] = meta['sample'].astype(str)

# 检查同一样本内临床信息是否一致
check_cols = [c for c in clinical_cols if c != 'sample']
consistency = meta.groupby('sample')[check_cols].nunique(dropna=False)
if (consistency > 1).any().any():
    raise ValueError(f"同一样本存在不一致的临床信息：\n{consistency[(consistency > 1).any(axis=1)]}")

meta = meta.drop_duplicates('sample').copy()

print("===== MPR 原始标签 =====")
print(meta['MPR'].value_counts(dropna=False))

print("\n===== 病理响应原始标签 =====")
print(meta['pathologic_response'].value_counts(dropna=False))

print("\n===== 时间点分布 =====")
print(meta['timepoint'].value_counts(dropna=False))

print("\n===== 样本临床信息 =====")
display(meta.sort_values(['patient', 'sample']))

# ============================================================
# 2. 合并模型预测结果
# ============================================================
if 'sample' not in pred.columns:
    pred = pred.reset_index()
    pred = pred.rename(columns={pred.columns[0]: 'sample'})

pred['sample'] = pred['sample'].astype(str)

if not pred['sample'].is_unique:
    raise ValueError("预测结果中存在重复 sample，请先检查预测结果。")

if 'Delta_Age' not in pred.columns:
    pred['Delta_Age'] = pred['Predicted_Age'] - pred['Age']

df_hnscc = pred.merge(meta, on='sample', how='left', validate='one_to_one', indicator=True)

if (df_hnscc['_merge'] != 'both').any():
    raise ValueError("部分预测样本未能匹配临床信息，请检查 sample 名称。")

df_hnscc = df_hnscc.drop(columns='_merge')

print("\n===== 合并结果 =====")
display(df_hnscc[['sample', 'patient', 'timepoint', 'Age',
                  'Predicted_Age', 'Delta_Age', 'MPR',
                  'pathologic_response', 'radiographic_response']])

print(f"\n预测样本数: {df_hnscc['sample'].nunique()}")
print(f"独立患者数: {df_hnscc['patient'].nunique()}")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import permutation_test

# ============================================================
# 4. MPR分组并选择P0样本
# ============================================================
df_hnscc['MPR_group'] = df_hnscc['MPR'].astype('string').str.strip().str.lower().map({
    'yes': 'MPR', 'no': 'nMPR'
})

df_compare = df_hnscc[
    (df_hnscc['timepoint'].astype(str) == 'P0') &
    df_hnscc['MPR_group'].notna()
].copy()

if df_compare['patient'].duplicated().any():
    raise ValueError("P0分析中存在重复患者，不能直接进行独立样本检验。")

print("===== P0 样本分组 =====")
print(df_compare['MPR_group'].value_counts())

display(df_compare[['sample', 'patient', 'timepoint', 'Age',
                    'Predicted_Age', 'Delta_Age', 'MPR_group']])

# ============================================================
# 5. 分组描述性统计与置换检验
# ============================================================
summary = df_compare.groupby('MPR_group')[['Age', 'Predicted_Age', 'Delta_Age']].agg(
    ['count', 'mean', 'median', 'std']
)
display(summary)

def compare_groups(data, metric):
    a = data.loc[data['MPR_group'] == 'MPR', metric].dropna().to_numpy()
    b = data.loc[data['MPR_group'] == 'nMPR', metric].dropna().to_numpy()

    print(f"\n===== {metric}: MPR vs nMPR =====")
    print(f"MPR: n={len(a)}, mean={np.mean(a):.3f}" if len(a) else "MPR: n=0")
    print(f"nMPR: n={len(b)}, mean={np.mean(b):.3f}" if len(b) else "nMPR: n=0")

    if min(len(a), len(b)) < 2:
        print("至少一组不足2名独立患者，仅进行描述性比较。")
        return np.nan

    res = permutation_test(
        (a, b),
        statistic=lambda x, y: np.mean(x) - np.mean(y),
        permutation_type='independent',
        n_resamples=np.inf,
        alternative='two-sided'
    )
    print(f"Mean difference (MPR - nMPR): {np.mean(a) - np.mean(b):.3f}")
    print(f"Exact permutation P = {res.pvalue:.4f}")
    return res.pvalue

p_age = compare_groups(df_compare, 'Age')
p_pred = compare_groups(df_compare, 'Predicted_Age')
p_delta = compare_groups(df_compare, 'Delta_Age')

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
import matplotlib.font_manager as fm
from scipy.stats import permutation_test

# ============================================================
# 1. Arial字体与全局参数
# ============================================================
font_dir = str(FONT_DIR)
for f in os.listdir(font_dir):
    if f.lower().endswith((".ttf", ".otf")):
        fm.fontManager.addfont(os.path.join(font_dir, f))

mpl.rcParams.update({
    "font.family": "Arial",
    "font.sans-serif": ["Arial"],
    "font.size": 15,
    "font.weight": "bold",
    "axes.labelsize": 17,
    "axes.labelweight": "bold",
    "axes.titlesize": 18,
    "axes.titleweight": "bold",
    "xtick.labelsize": 14,
    "ytick.labelsize": 14,
    "axes.linewidth": 1.5,
    "xtick.major.width": 1.3,
    "ytick.major.width": 1.3,
    "xtick.major.size": 5,
    "ytick.major.size": 5,
    "pdf.fonttype": 42,
    "ps.fonttype": 42
})

# ============================================================
# 2. 整理P0数据
# ============================================================
df_plot = df_hnscc.copy()
df_plot["MPR_group"] = df_plot["MPR"].astype(str).str.strip().str.lower().map({
    "yes": "MPR",
    "no": "nMPR"
})
df_plot["timepoint"] = df_plot["timepoint"].astype(str).str.strip().str.upper()
df_plot = df_plot[(df_plot["timepoint"] == "P0") & (df_plot["MPR_group"].notna())].copy()

if df_plot["patient"].duplicated().any():
    raise ValueError("P0 中存在重复 patient，请先检查数据。")

print(df_plot[["sample", "patient", "Age", "Predicted_Age", "Delta_Age", "MPR_group"]])

# ============================================================
# 3. 置换检验
# ============================================================
a = df_plot.loc[df_plot["MPR_group"] == "MPR", "Delta_Age"].dropna().to_numpy()
b = df_plot.loc[df_plot["MPR_group"] == "nMPR", "Delta_Age"].dropna().to_numpy()

if min(len(a), len(b)) >= 2:
    res = permutation_test(
        (a, b),
        statistic=lambda x, y: np.mean(x) - np.mean(y),
        permutation_type="independent",
        n_resamples=np.inf,
        alternative="two-sided"
    )
    p_delta = res.pvalue
else:
    p_delta = np.nan

print(f"Permutation P (Delta_Age) = {p_delta}")

# ============================================================
# 4. 作图函数：顶部括号
# ============================================================
def add_pvalue_bracket(ax, x1, x2, y, h, text, lw=1.8, fs=14):
    """
    x1, x2: 两组x坐标
    y: 括号底部高度
    h: 括号高度
    """
    ax.plot([x1, x1, x2, x2], [y, y+h, y+h, y],
            color="black", lw=lw, clip_on=False)
    ax.text((x1 + x2) / 2, y + h + 0.15, text,
            ha="center", va="bottom",
            fontsize=fs, fontweight="bold")

# ============================================================
# 5. 作图
# ============================================================
# 颜色按你给的示意图改：MPR粉色，nMPR蓝色
colors = {"MPR": "#C97D8A", "nMPR": "#6E95AE"}
groups = ["MPR", "nMPR"]
rng = np.random.default_rng(42)

fig, ax = plt.subplots(figsize=(4.6, 5.2), dpi=300)

# 两组的横坐标位置
xpos = {"MPR": 0, "nMPR": 1}

# 逐组画散点和均值横线
for group in groups:
    sub = df_plot[df_plot["MPR_group"] == group].copy()
    sub = sub.sort_values("Delta_Age").reset_index(drop=True)

    values = sub["Delta_Age"].to_numpy()
    x_center = xpos[group]

    # 散点横向轻微抖动
    x = x_center + rng.uniform(-0.04, 0.04, len(values))

    # 散点
    ax.scatter(
        x, values,
        s=190,
        color=colors[group],
        edgecolor="black",
        linewidth=1.1,
        zorder=3
    )

    # 均值横线
    mean_val = np.mean(values)
    ax.hlines(
        y=mean_val,
        xmin=x_center - 0.18,
        xmax=x_center + 0.18,
        colors="black",
        linewidth=2.2,
        zorder=4
    )

    # 标注 patient 名称
    for xi, (_, row) in zip(x, sub.iterrows()):
        yv = row["Delta_Age"]
        label = str(row["patient"])

        # 根据点的上下位置自动调整文字偏移
        if yv >= mean_val:
            offset = (5, 8)
            va = "bottom"
        else:
            offset = (5, -8)
            va = "top"

        ax.annotate(
            label,
            (xi, yv),
            xytext=offset,
            textcoords="offset points",
            ha="left",
            va=va,
            fontsize=11,
            fontweight="bold"
        )

# y=0 虚线
ax.axhline(0, color="gray", linestyle="--", linewidth=1.4, alpha=0.9, zorder=1)

# 顶部P值括号
ymax = df_plot["Delta_Age"].max()
ymin = df_plot["Delta_Age"].min()
yrange = ymax - ymin

bracket_y = ymax + 0.08 * yrange
bracket_h = 0.02 * yrange

if np.isfinite(p_delta):
    p_text = f"P = {p_delta:.3f}"
else:
    p_text = "P = NA"

add_pvalue_bracket(
    ax,
    x1=xpos["MPR"],
    x2=xpos["nMPR"],
    y=bracket_y,
    h=bracket_h,
    text=p_text,
    lw=1.8,
    fs=14
)

# 坐标轴设置
ax.set_xticks([xpos["MPR"], xpos["nMPR"]])
ax.set_xticklabels(["MPR", "nMPR"], fontweight="bold")

ax.set_ylabel("Age difference (Predicted - Actual)", fontweight="bold")
ax.set_title("Baseline PBMC immune age", pad=10, fontweight="bold")

ax.set_xlim(-0.4, 1.4)
ax.set_ylim(ymin - 0.8, bracket_y + bracket_h + 0.8)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.tick_params(axis="both", width=1.3)

plt.tight_layout(pad=0.6)

# ============================================================
# 6. 保存图片
# ============================================================
out_png = str(OUTPUT_DIR / "fig/bulk_perigenes_hnscc_respond_scatter.png")
out_pdf = str(OUTPUT_DIR / "fig/bulk_perigenes_hnscc_respond_scatter.pdf")
os.makedirs(os.path.dirname(out_png), exist_ok=True)

fig.savefig(out_png, dpi=300, bbox_inches="tight", pad_inches=0.05)
fig.savefig(out_pdf, bbox_inches="tight", pad_inches=0.05)
plt.show()

print("Saved to:")
print(out_png)
print(out_pdf)

## Gene coefficient weights (Extended Data Fig. 10)

Source: cells 51–54 and 58. PeriClock is read from the supplied final artifact; the bulk pipeline and feature metadata are read from the newly generated output files above. Original absolute-coefficient aggregation, relative weights, feature parsing and plotting are preserved. This comparison is not a new age predictor. No section of this notebook was run during preparation.

In [ ]:
# 提取两个模型系数

import os, json, joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
import matplotlib.font_manager as fm

# ============================================================
# 1. 路径与字体
# ============================================================
peri_path = str(MODEL_PATH)
model_dir = str(OUTPUT_DIR / "model")
outdir = "model_weight_comparison"
os.makedirs(outdir, exist_ok=True)

font_dir = str(FONT_DIR)
if os.path.isdir(font_dir):
    for f in os.listdir(font_dir):
        if f.lower().endswith((".ttf", ".otf")):
            fm.fontManager.addfont(os.path.join(font_dir, f))

mpl.rcParams.update({
    "font.family": "Arial", "font.size": 12,
    "axes.linewidth": 1.2, "pdf.fonttype": 42, "ps.fonttype": 42
})

# ============================================================
# 2. PeriClock
# ============================================================
pipeline = joblib.load(peri_path)

vt = pipeline["vt"]
skb = pipeline["skb"]
scaler_prop = pipeline["scaler_prop"]
scaler_expr = pipeline["scaler_expr"]
peri_model = pipeline["model"]

prop_cols = list(map(str, pipeline["prop_cols"]))
expr_cols = list(map(str, pipeline["expr_cols"]))
peri_features = list(map(str, pipeline["selected_features"]))
peri_coef = np.asarray(peri_model.coef_, dtype=float).ravel()

if len(peri_features) != len(peri_coef):
    raise ValueError(
        f"PeriClock 特征与系数不匹配："
        f"selected_features={len(peri_features)}, coef={len(peri_coef)}。"
        "需要检查训练时的筛选和特征拼接顺序。"
    )

peri_df = pd.DataFrame({
    "feature": peri_features,
    "coef": peri_coef
})

expr_set, prop_set = set(expr_cols), set(prop_cols)
peri_df["feature_type"] = np.select(
    [peri_df["feature"].isin(expr_set), peri_df["feature"].isin(prop_set)],
    ["expression", "proportion"], default="unknown"
)

unknown = peri_df[peri_df["feature_type"] == "unknown"]
if len(unknown):
    print("无法确定类型的 PeriClock 特征：")
    print(unknown.head(20))
    raise ValueError("请先核对 selected_features 与 expr_cols / prop_cols 的命名。")

print("PeriClock 总特征数:", len(peri_df))
print("表达特征数:", (peri_df.feature_type == "expression").sum())
print("比例特征数:", (peri_df.feature_type == "proportion").sum())
print("非零系数数:", np.count_nonzero(peri_coef))

# ============================================================
# 3. Bulk-perigenes clock
# ============================================================
bulk_model = joblib.load(os.path.join(model_dir, "bulk_clock_perigenes_model.pkl"))

with open(os.path.join(model_dir, "bulk_clock_perigenes_model_info.json")) as f:
    model_info = json.load(f)

scaler = bulk_model.named_steps["standardscaler"]
en = bulk_model.named_steps["elasticnetcv"]

all_genes = list(map(str, model_info["feature_names"]))
bulk_coef = np.asarray(en.coef_, dtype=float).ravel()

if len(all_genes) != len(bulk_coef) or len(all_genes) != len(scaler.scale_):
    raise ValueError("Bulk 模型的特征顺序、回归系数和 scaler 维度不一致。")

if not pd.Index(all_genes).is_unique:
    raise ValueError("Bulk feature_names 存在重复基因，请检查。")

bulk_df = pd.DataFrame({"gene": all_genes, "bulk_coef": bulk_coef})
bulk_df["bulk_abs_coef"] = bulk_df["bulk_coef"].abs()

print("Bulk 候选基因数:", len(bulk_df))
print("Bulk 非零系数基因数:", np.count_nonzero(bulk_coef))


In [ ]:
# 将PeriClock特征聚集到基因层面

# ============================================================
# 4. 解析 PeriClock 的 celltype-gene
# ============================================================
peri_expr = peri_df.loc[
    peri_df.feature_type == "expression", ["feature", "coef"]
].copy()

gene_candidates = sorted(all_genes, key=len, reverse=True)

def parse_feature(feature):
    for gene in gene_candidates:
        suffix = "_" + gene
        if feature.endswith(suffix):
            return pd.Series({
                "cell_type": feature[:-len(suffix)],
                "gene": gene
            })
    return pd.Series({"cell_type": None, "gene": None})

peri_expr[["cell_type", "gene"]] = peri_expr["feature"].apply(parse_feature)

unmatched = peri_expr[peri_expr["gene"].isna()].copy()
unmatched.to_csv(os.path.join(outdir, "unmatched_periclock_features.csv"), index=False)

print("成功匹配的表达特征数:", peri_expr["gene"].notna().sum())
print("未匹配特征数:", len(unmatched))

if len(unmatched):
    print("未匹配的前20个特征（请检查是否属于 bulk 候选基因）：")
    print(unmatched["feature"].head(20).tolist())

matched = peri_expr.dropna(subset=["gene"]).copy()
matched["abs_coef"] = matched["coef"].abs()

matched.to_csv(
    os.path.join(outdir, "periclock_celltype_gene_coefficients.csv"),
    index=False
)
bulk_df.to_csv(os.path.join(outdir, "bulk_gene_coefficients.csv"), index=False)

# ============================================================
# 5. 聚合到 gene level
# ============================================================
peri_gene = matched.groupby("gene").agg(
    peri_sum_abs=("abs_coef", "sum"),
    peri_max_abs=("abs_coef", "max"),
    peri_mean_abs=("abs_coef", "mean"),
    peri_signed_sum=("coef", "sum"),
    n_celltypes=("cell_type", "nunique"),
    n_nonzero=("coef", lambda x: np.count_nonzero(x))
)

# 方向抵消指数：0 = 同方向；接近1 = 正负系数明显并存
peri_gene["cancellation_index"] = np.where(
    peri_gene["peri_sum_abs"] > 0,
    1 - peri_gene["peri_signed_sum"].abs() /
        peri_gene["peri_sum_abs"].replace(0, np.nan),
    0
)

# ============================================================
# 6. 对比共同基因
# ============================================================
comparison = peri_gene.join(
    bulk_df.set_index("gene")[["bulk_coef", "bulk_abs_coef"]],
    how="inner"
)

if comparison.empty:
    raise ValueError("没有匹配到共同基因，请检查 PeriClock 特征命名。")

peri_total = comparison["peri_sum_abs"].sum()
bulk_total = comparison["bulk_abs_coef"].sum()

if peri_total == 0 or bulk_total == 0:
    raise ValueError("共同基因的系数绝对值之和为0，无法计算相对权重。")

comparison["peri_relative"] = comparison["peri_sum_abs"] / peri_total
comparison["bulk_relative"] = comparison["bulk_abs_coef"] / bulk_total
comparison["delta_relative"] = (
    comparison["peri_relative"] - comparison["bulk_relative"]
)

comparison["peri_rank"] = comparison["peri_relative"].rank(
    ascending=False, method="min"
).astype(int)

comparison["bulk_rank"] = comparison["bulk_relative"].rank(
    ascending=False, method="min"
).astype(int)

comparison["rank_gain_peri"] = comparison["bulk_rank"] - comparison["peri_rank"]

comparison["weight_direction"] = np.select(
    [comparison.delta_relative > 0, comparison.delta_relative < 0],
    ["PeriClock_higher", "Bulk_higher"], default="Equal"
)

comparison = comparison.sort_values("delta_relative", ascending=False)
comparison.to_csv(os.path.join(outdir, "gene_weight_comparison.csv"))

print("\n共同基因数:", len(comparison))
print("\nPeriClock 相对权重更高的前20个基因:")
print(comparison.head(20).round(5).to_string())

print("\nBulk 相对权重更高的前20个基因:")
print(comparison.tail(20).sort_values("delta_relative").round(5).to_string())


In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
import matplotlib.font_manager as fm
from adjustText import adjust_text

# ============================================================
# 1. 路径与 Arial 字体
# ============================================================
outdir = str(OUTPUT_DIR / "fig")
os.makedirs(outdir, exist_ok=True)

font_dir = str(FONT_DIR)
if os.path.isdir(font_dir):
    for f in os.listdir(font_dir):
        if f.lower().endswith((".ttf", ".otf")): fm.fontManager.addfont(os.path.join(font_dir, f))

mpl.rcParams.update({
    "font.family": "Arial", "font.sans-serif": ["Arial"], "font.size": 14, "font.weight": "bold",
    "axes.labelsize": 17, "axes.labelweight": "bold", "axes.titlesize": 19, "axes.titleweight": "bold",
    "xtick.labelsize": 13, "ytick.labelsize": 13, "axes.linewidth": 1.4,
    "xtick.major.width": 1.2, "ytick.major.width": 1.2,
    "pdf.fonttype": 42, "ps.fonttype": 42
})

# ============================================================
# Fig A. Gene relative weight scatter
# ============================================================
fig, ax = plt.subplots(figsize=(9, 8))

x = comparison["bulk_relative"] * 100
y = comparison["peri_relative"] * 100
colors = np.where(comparison["delta_relative"] >= 0, "#86BCD8", "#E6A6AC")

ax.scatter(x, y, c=colors, s=65, alpha=0.8, edgecolors="white", linewidths=0.5, zorder=2)

lim = max(x.max(), y.max()) * 1.12
ax.plot([0, lim], [0, lim], "--", color="#888888", lw=1.3, zorder=1)
ax.set_xlim(-lim * 0.03, lim)
ax.set_ylim(-lim * 0.03, lim)

# 标注差异最大的12个基因
top_labels = comparison["delta_relative"].abs().nlargest(12).index
texts = []

for gene in top_labels:
    txt = ax.text(x.loc[gene], y.loc[gene], gene, fontsize=12, fontweight="bold", color="#222222", ha="center", va="center", zorder=5)
    texts.append(txt)

# 自动避让基因标签，添加连接线
adjust_text(
    texts, ax=ax, x=x.to_numpy(), y=y.to_numpy(),
    expand=(1.2, 1.5), force_text=(0.8, 1.0), force_static=(0.5, 0.7),
    arrowprops=dict(arrowstyle="-", color="#777777", lw=0.9),
    ensure_inside_axes=True
)

ax.set_xlabel("Bulk relative coefficient weight (%)", fontsize=17, fontweight="bold", labelpad=12)
ax.set_ylabel("PeriClock relative coefficient weight (%)", fontsize=17, fontweight="bold", labelpad=12)
ax.set_title("Gene weight comparison", fontsize=19, fontweight="bold", pad=18)

ax.tick_params(axis="both", labelsize=13, width=1.2, length=5)
for tick in ax.get_xticklabels() + ax.get_yticklabels(): tick.set_fontweight("bold")

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.grid(False)

plt.tight_layout()
fig.savefig(os.path.join(outdir, "gene_weight_scatter.pdf"), bbox_inches="tight", pad_inches=0.2)
fig.savefig(os.path.join(outdir, "gene_weight_scatter.png"), dpi=600, bbox_inches="tight", pad_inches=0.2)
plt.show()

# ============================================================
# Fig B. Genes with largest relative weight changes
# ============================================================
n = min(12, len(comparison))
top_peri = comparison.nlargest(n, "delta_relative")
top_bulk = comparison.nsmallest(n, "delta_relative")
plot_df = pd.concat([top_peri, top_bulk])
plot_df = plot_df.loc[~plot_df.index.duplicated()].sort_values("delta_relative")

fig, ax = plt.subplots(figsize=(10, max(7, len(plot_df) * 0.42)))

bar_colors = np.where(plot_df["delta_relative"] >= 0, "#86BCD8", "#E6A6AC")
values = plot_df["delta_relative"] * 100

ax.barh(plot_df.index, values, color=bar_colors, edgecolor="none", height=0.72, zorder=2)
ax.axvline(0, color="#333333", linewidth=1.3, zorder=3)

ax.set_xlabel("Relative coefficient weight difference (percentage points)", fontsize=17, fontweight="bold", labelpad=14)
ax.set_title("Genes with largest weight shifts", fontsize=19, fontweight="bold", pad=18)

ax.tick_params(axis="x", labelsize=13, width=1.2, length=5)
ax.tick_params(axis="y", labelsize=12, length=0, pad=8)

for tick in ax.get_xticklabels() + ax.get_yticklabels(): tick.set_fontweight("bold")

# 增加左右留白，避免柱子贴近边缘
max_abs = max(abs(values.min()), abs(values.max()))
ax.set_xlim(-max_abs * 1.15, max_abs * 1.15)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.spines["left"].set_visible(False)
ax.spines["bottom"].set_linewidth(1.4)
ax.grid(False)

plt.tight_layout()
fig.savefig(os.path.join(outdir, "gene_weight_shift.pdf"), bbox_inches="tight", pad_inches=0.2)
fig.savefig(os.path.join(outdir, "gene_weight_shift.png"), dpi=600, bbox_inches="tight", pad_inches=0.2)
plt.show()

print("All figures saved to:", outdir)

In [ ]:
# ============================================================
# 7. 为每个 gene 整理其在 PeriClock 中对应的 cell type 信息
# ============================================================
def summarize_celltypes(df, top_n=3):
    """
    df: 某一个 gene 在 matched 中对应的所有 cell_type-gene 行
    top_n: 标签里最多展示几个 cell type
    """
    d = df.sort_values("abs_coef", ascending=False).copy()
    
    celltypes = d["cell_type"].tolist()
    n_all = len(celltypes)
    top_ct = celltypes[:top_n]
    
    if n_all > top_n:
        label = ", ".join(top_ct) + f" (+{n_all - top_n})"
    else:
        label = ", ".join(top_ct)
    
    # 也保留更详细的信息，后面可以导出表格
    return pd.Series({
        "n_celltypes_total": d["cell_type"].nunique(),
        "top_celltypes_label": label,
        "all_celltypes": "; ".join(celltypes),
        "top1_celltype": celltypes[0] if len(celltypes) >= 1 else None,
        "top2_celltype": celltypes[1] if len(celltypes) >= 2 else None,
        "top3_celltype": celltypes[2] if len(celltypes) >= 3 else None
    })

gene_ct_info = matched.groupby("gene").apply(summarize_celltypes, top_n=3)

comparison_plot = comparison.join(gene_ct_info, how="left")

comparison_plot.to_csv(
    os.path.join(outdir, "gene_weight_comparison_with_celltypes.csv")
)

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
import matplotlib.font_manager as fm
from matplotlib.lines import Line2D

# ============================================================
# 0. 全局设置
# ============================================================
outdir = "model_weight_comparison"
os.makedirs(outdir, exist_ok=True)

font_dir = str(FONT_DIR)
if os.path.isdir(font_dir):
    for f in os.listdir(font_dir):
        if f.lower().endswith((".ttf", ".otf")):
            fm.fontManager.addfont(os.path.join(font_dir, f))

mpl.rcParams.update({
    "font.family": "Arial",
    "font.sans-serif": ["Arial"],
    "font.size": 12,
    "axes.linewidth": 1.2,
    "pdf.fonttype": 42,
    "ps.fonttype": 42
})

# ============================================================
# 1. 准备 gene -> cell type 信息
# 需要已有：
# matched: gene, cell_type, coef, abs_coef
# comparison: index=gene，含 peri_relative, bulk_relative, delta_relative
# ============================================================
def summarize_celltypes(df):
    d = df.sort_values("abs_coef", ascending=False).copy()
    celltypes = d["cell_type"].tolist()
    return pd.Series({
        "n_celltypes_total": d["cell_type"].nunique(),
        "top1_celltype": celltypes[0] if len(celltypes) >= 1 else "",
        "top2_celltype": celltypes[1] if len(celltypes) >= 2 else "",
        "top3_celltype": celltypes[2] if len(celltypes) >= 3 else "",
        "all_celltypes": "; ".join(celltypes)
    })

gene_ct_info = matched.groupby("gene").apply(summarize_celltypes)
comparison_plot = comparison.join(gene_ct_info, how="left")

# ============================================================
# 2. 选择高亮基因
# ============================================================
highlight_n = 10
highlight_genes = comparison_plot["delta_relative"].abs().nlargest(highlight_n).index
highlight_df = comparison_plot.loc[highlight_genes].copy()

comparison_plot["x"] = comparison_plot["bulk_relative"] * 100
comparison_plot["y"] = comparison_plot["peri_relative"] * 100

highlight_df["x"] = highlight_df["bulk_relative"] * 100
highlight_df["y"] = highlight_df["peri_relative"] * 100

highlight_df["side"] = np.where(highlight_df["delta_relative"] >= 0, "left", "right")

# 正文图建议只显示 top1 cell type
highlight_df["label"] = [
    f"{gene}\n{row['top1_celltype']}" if pd.notna(row["top1_celltype"]) and row["top1_celltype"] != "" else gene
    for gene, row in highlight_df.iterrows()
]

# ============================================================
# 3. 自动分配标签纵向位置
# ============================================================
def distribute_positions(y_values, y_min, y_max, min_gap):
    y_values = np.array(y_values, dtype=float)
    n = len(y_values)
    if n == 0:
        return np.array([])

    order = np.argsort(-y_values)  # 从高到低
    y_sorted = y_values[order].copy()

    y_adj = y_sorted.copy()
    y_adj[0] = min(max(y_adj[0], y_min), y_max)

    for i in range(1, n):
        upper_allowed = y_adj[i-1] - min_gap
        y_adj[i] = min(y_adj[i], upper_allowed)

    if y_adj[-1] < y_min:
        shift = y_min - y_adj[-1]
        y_adj += shift
        if y_adj[0] > y_max:
            if n == 1:
                y_adj[0] = np.clip(y_sorted[0], y_min, y_max)
            else:
                available = y_max - y_min
                need = min_gap * (n - 1)
                if need <= available:
                    y_adj = np.linspace(y_max, y_max - need, n)
                else:
                    y_adj = np.linspace(y_max, y_min, n)

    y_adj = np.clip(y_adj, y_min, y_max)

    result = np.empty_like(y_adj)
    result[order] = y_adj
    return result

# ============================================================
# 4. 作图
# ============================================================
fig, ax = plt.subplots(figsize=(10, 7.2))

# 背景点
bg_colors = np.where(
    comparison_plot["delta_relative"] >= 0,
    "#9ecae1",
    "#f2b6b6"
)

ax.scatter(
    comparison_plot["x"], comparison_plot["y"],
    c=bg_colors, s=36, alpha=0.75,
    edgecolors="white", linewidths=0.35, zorder=1
)

# 高亮点
left_color = "#2b8cbe"
right_color = "#d95f5f"
left_line = "#86BCD8"
right_line = "#E6A6AC"

for gene, row in highlight_df.iterrows():
    c = left_color if row["side"] == "left" else right_color
    ax.scatter(
        row["x"], row["y"],
        s=135, c=c,
        edgecolors="black", linewidths=0.9, zorder=4
    )

# 对角线
lim = max(comparison_plot["x"].max(), comparison_plot["y"].max()) * 1.08
ax.plot([0, lim], [0, lim], "--", color="gray", lw=1.0, zorder=0)

ax.set_xlim(0, lim)
ax.set_ylim(0, lim)

# ============================================================
# 5. 左右标签布局
# ============================================================
left_df = highlight_df[highlight_df["side"] == "left"].copy().sort_values("y", ascending=False)
right_df = highlight_df[highlight_df["side"] == "right"].copy().sort_values("y", ascending=False)

# 更合理的纵向范围
y_min_left, y_max_left = lim * 0.36, lim * 0.93
y_min_right, y_max_right = lim * 0.22, lim * 0.80

min_gap_left = lim * 0.070
min_gap_right = lim * 0.070

left_df["y_text"] = distribute_positions(
    left_df["y"].values, y_min=y_min_left, y_max=y_max_left, min_gap=min_gap_left
)
right_df["y_text"] = distribute_positions(
    right_df["y"].values, y_min=y_min_right, y_max=y_max_right, min_gap=min_gap_right
)

# 左右标签横坐标：左边更往外放，避免压到 ylabel
x_left_text = -lim * 0.20
x_right_text = lim * 1.12

# 左边留更大边距；右边也稍微多留一点
fig.subplots_adjust(left=0.30, right=0.80)

# 把 y 轴标题往轴线内侧拉一点，避免跟左侧标签冲突
ax.yaxis.set_label_coords(-0.06, 0.5)

# ============================================================
# 6. 折线 + 标签
# ============================================================
def draw_connector_label(ax, x0, y0, x_text, y_text, text, side,
                         linecolor, fontsize_gene=10, fontsize_ct=9):
    if side == "left":
        x_knee = x0 - (x0 - x_text) * 0.42
        ha = "right"
        text_x = x_text
    else:
        x_knee = x0 + (x_text - x0) * 0.42
        ha = "left"
        text_x = x_text

    # 折线
    ax.plot([x0, x_knee], [y0, y0], color=linecolor, lw=0.9, alpha=0.9, zorder=2, clip_on=False)
    ax.plot([x_knee, x_knee], [y0, y_text], color=linecolor, lw=0.9, alpha=0.9, zorder=2, clip_on=False)
    ax.plot([x_knee, text_x], [y_text, y_text], color=linecolor, lw=0.9, alpha=0.9, zorder=2, clip_on=False)

    lines = str(text).split("\n")
    gene_line = lines[0]
    ct_line = lines[1] if len(lines) > 1 else ""

    ax.text(
        text_x, y_text + lim * 0.008,
        gene_line,
        ha=ha, va="bottom",
        fontsize=fontsize_gene, fontweight="bold",
        color="black",
        zorder=5, clip_on=False
    )

    if ct_line != "":
        ax.text(
            text_x, y_text - lim * 0.002,
            ct_line,
            ha=ha, va="top",
            fontsize=fontsize_ct,
            color="black",
            zorder=5, clip_on=False
        )

# 左侧
for gene, row in left_df.iterrows():
    draw_connector_label(
        ax=ax,
        x0=row["x"], y0=row["y"],
        x_text=x_left_text, y_text=row["y_text"],
        text=row["label"],
        side="left",
        linecolor=left_line
    )

# 右侧
for gene, row in right_df.iterrows():
    draw_connector_label(
        ax=ax,
        x0=row["x"], y0=row["y"],
        x_text=x_right_text, y_text=row["y_text"],
        text=row["label"],
        side="right",
        linecolor=right_line
    )

# ============================================================
# 7. 坐标轴与图例
# ============================================================
ax.set_xlabel("Bulk relative coefficient weight (%)", fontsize=16, fontweight="bold", labelpad=10)
ax.set_ylabel("PeriClock relative coefficient weight (%)", fontsize=16, fontweight="bold", labelpad=6)
ax.set_title("Gene weight comparison", fontsize=17, fontweight="bold", pad=10)

ax.tick_params(axis="both", labelsize=12)

legend_elements = [
    Line2D([0], [0], marker='o', color='w',
           label='PeriClock-enriched weight',
           markerfacecolor=left_color, markeredgecolor='black', markersize=9),
    Line2D([0], [0], marker='o', color='w',
           label='Bulk-enriched weight',
           markerfacecolor=right_color, markeredgecolor='black', markersize=9)
]
ax.legend(handles=legend_elements, frameon=False, loc="upper left", fontsize=11)

plt.tight_layout()

fig.savefig(os.path.join(outdir, "gene_weight_scatter_maintext_optimized.pdf"),
            bbox_inches="tight")
fig.savefig(os.path.join(outdir, "gene_weight_scatter_maintext_optimized.png"),
            dpi=300, bbox_inches="tight")

plt.show()